# Power BI Dashboard Preparation

**Objective:** Create KPI, calendar, monthly summary, growth, correlation, anomaly, and forecast tables for dashboard use.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from rbi_utils import *

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 120)


In [ ]:
df = load_cleaned()
processed = PROJECT_ROOT / 'data' / 'processed'
calendar = pd.DataFrame({'date': pd.date_range(df['date'].min(), df['date'].max(), freq='D')})
calendar['year'] = calendar['date'].dt.year
calendar['month'] = calendar['date'].dt.month
calendar['quarter'] = calendar['date'].dt.quarter
calendar['financial_year'] = np.where(calendar['month'] >= 4, calendar['year'].astype(str) + '-' + (calendar['year'] + 1).astype(str).str[-2:], (calendar['year'] - 1).astype(str) + '-' + calendar['year'].astype(str).str[-2:])
calendar.to_csv(processed / 'calendar_dimension.csv', index=False)
calendar.head()

In [ ]:
latest = df.iloc[-1]
kpis = pd.DataFrame({'metric': ['latest_date', 'reserve_money', 'weekly_growth_pct', 'net_foreign_exchange_assets', 'currency_in_circulation'], 'value': [latest['date'], latest['reserve_money'], latest['reserve_money_weekly_growth_pct'], latest['net_foreign_exchange_assets'], latest['currency_in_circulation']]})
kpis.to_csv(processed / 'dashboard_kpis.csv', index=False)
kpis

In [ ]:
corr_long = df[KEY_SERIES].corr().reset_index().melt(id_vars='index', var_name='variable_2', value_name='correlation').rename(columns={'index': 'variable_1'})
corr_long.to_csv(processed / 'correlation_long.csv', index=False)
corr_long.head()

In [ ]:
anomalies = df.loc[df['reserve_money_zscore'].abs() > 3, ['date', 'reserve_money', 'reserve_money_zscore', 'reserve_money_weekly_growth_pct']]
anomalies.to_csv(processed / 'zscore_anomalies.csv', index=False)
anomalies.head()